# 08 — Validate model inputs and assumptions

This notebook constructs the **exact analysis samples** used in notebook 09 and checks the assumptions that can be assessed before estimation. It does not fit or plot the final models.

The founding analysis is an NB2 count model for quarterly births in 100 m cells (2016–2025, with 2015 used only to form year-on-year population growth). The survival analysis is a Fachgruppe-stratified Cox model on firm-age time with left truncation, time-varying quarterly covariates, and 100 m cell-clustered inference.

Run order:

1. this notebook: schemas, samples, transformations, collinearity, event support;
2. `09_estimate_and_save_models.ipynb`: estimation and portable result export;
3. `10_visualize_model_results.ipynb`: read-only loading and figures.

Some assumptions are inherently post-estimation. The NB2 boundary LR test and Cox proportional-hazards diagnostics are therefore calculated in notebook 09 and reviewed in notebook 10.


## Configuration


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def discover_project_dir() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "ANAL").is_dir() and (candidate / "OGD").is_dir():
            return candidate
    raise RuntimeError("Run from the project directory or one of its subdirectories.")


PROJECT_DIR = discover_project_dir()
if str(PROJECT_DIR / "ANAL") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "ANAL"))

import model_workflow as mw

PATHS = mw.project_paths(PROJECT_DIR)
PATHS.models.mkdir(parents=True, exist_ok=True)

CORRELATION_THRESHOLD = 0.70



## 1. Input preflight


In [ ]:
mw.preflight_founding(PATHS)
mw.preflight_survival(PATHS)
print(f"Founding inputs: {mw.LAG_YEAR}–{mw.END_YEAR}; analysis starts {mw.START_YEAR}.")
print(f"Survival inputs: {mw.START_YEAR}–{mw.END_YEAR}.")
print("All required files and columns are present.")


## 2. Founding-model sample

The two highly correlated neighbourhood stocks are reparameterized as population mass and a relative firm-density index:

`log(1 + firms) - log(1 + population)`.

This is an exact linear reparameterization of the two logged stocks. It is **not** a literal firms-per-capita rate because of the `+1` pseudocount. Notebook 09 transforms the coefficients back to separate population and firm-stock effects for interpretation.


In [ ]:
founding, X_founding, y_founding, founding_clusters, founding_terms, ring_specs = mw.founding_model_data(PATHS)
print(f"Rows: {len(founding):,}")
print(f"Cells: {founding['grid_id'].nunique():,}")
print(f"Quarters: {founding['period'].nunique()}")
print(f"Births: {int(y_founding.sum()):,}")
print(f"Zero share: {(y_founding == 0).mean():.2%}")


In [ ]:
founding_correlations = mw.high_correlations(founding, founding_terms, CORRELATION_THRESHOLD)
founding_condition = mw.standardized_condition_number(founding, founding_terms)

mean_births = float(y_founding.mean())
variance_births = float(y_founding.var())
founding_diagnostics = pd.Series({
    "n_observations": len(founding),
    "n_cells": founding['grid_id'].nunique(),
    "n_quarters": founding['period'].nunique(),
    "n_births": int(y_founding.sum()),
    "outcome_mean": mean_births,
    "outcome_variance": variance_births,
    "variance_to_mean": variance_births / mean_births if mean_births else np.nan,
    "zero_share": float((y_founding == 0).mean()),
    "births_with_zero_lagged_stock": int(((y_founding > 0) & (founding['active_firms_tminus1'] == 0)).sum()),
    "n_clusters": founding_clusters.nunique(),
    "standardized_condition_number": founding_condition,
})
display(founding_diagnostics.to_frame("value"))
display(founding_correlations if len(founding_correlations) else pd.DataFrame({"result": ["No pair at or above the threshold."]}))

founding_diagnostics.to_csv(PATHS.models / "founding_input_diagnostics.csv", header=["value"])
founding_correlations.to_csv(PATHS.models / "founding_high_correlations.csv", index=False)


### Founding checks to interpret

- Counts must be non-negative integers; this is enforced by the shared preparation code.
- A variance-to-mean ratio above one is only descriptive evidence of overdispersion. Notebook 09 performs the boundary LR test against Poisson.
- Period indicators absorb common quarter shocks. Standard errors are clustered by cell for repeated observations, but residual spatial dependence between cells remains a limitation until a spatial diagnostic is added.
- Births where lagged stock is zero show why `log1p(stock)` cannot automatically be used as a strict exposure offset. The offset restriction is tested as a nested robustness model in notebook 09.


## 3. Survival-model sample


In [ ]:
spells, survival_frame, survival_terms = mw.survival_model_data(PATHS)
first_intervals = survival_frame.groupby("standort_id", sort=False).first()
events_by_stratum = survival_frame.groupby("Fachgruppe_ID")["event"].agg(["sum", "size"])

print(f"Intervals: {len(survival_frame):,}")
print(f"Locations: {survival_frame['standort_id'].nunique():,}")
print(f"Observed exits: {int(survival_frame['event'].sum()):,}")
print(f"Left-truncated locations: {(first_intervals['start'] > 0).mean():.2%}")
print(f"Fachgruppe strata without an exit: {(events_by_stratum['sum'] == 0).sum()}")


In [ ]:
survival_correlations = mw.high_correlations(survival_frame, survival_terms, CORRELATION_THRESHOLD)
survival_condition = mw.standardized_condition_number(survival_frame, survival_terms)

sector_mapping = (
    survival_frame[["Fachgruppe_ID", "sparte", "sparte_name"]]
    .drop_duplicates()
    .sort_values(["sparte", "Fachgruppe_ID"])
)
ambiguous_groups = sector_mapping.groupby("Fachgruppe_ID")["sparte"].nunique().gt(1).sum()

survival_diagnostics = pd.Series({
    "n_intervals": len(survival_frame),
    "n_locations": survival_frame['standort_id'].nunique(),
    "n_events": int(survival_frame['event'].sum()),
    "n_cells": survival_frame['grid_id'].nunique(),
    "n_fachgruppe_strata": len(events_by_stratum),
    "strata_without_events": int((events_by_stratum['sum'] == 0).sum()),
    "strata_below_30_events": int((events_by_stratum['sum'] < 30).sum()),
    "left_truncated_location_share": float((first_intervals['start'] > 0).mean()),
    "events_per_covariate": float(survival_frame['event'].sum() / len(survival_terms)),
    "standardized_condition_number": survival_condition,
    "ambiguous_fachgruppe_sector_mappings": int(ambiguous_groups),
})
display(survival_diagnostics.to_frame("value"))
display(survival_correlations if len(survival_correlations) else pd.DataFrame({"result": ["No pair at or above the threshold."]}))
display(events_by_stratum.sort_values("sum").head(15))

survival_diagnostics.to_csv(PATHS.models / "survival_input_diagnostics.csv", header=["value"])
survival_correlations.to_csv(PATHS.models / "survival_high_correlations.csv", index=False)
sector_mapping.to_csv(PATHS.models / "official_sector_mapping.csv", index=False)


## 4. Decision before estimation

Proceed only if the diagnostics above describe the intended population and no assertion failed.

- Firm age is the Cox time scale. Locations already active at the first observed quarter enter at their attained age, which implements left truncation.
- Fachgruppe is used for strata; the official `Sparte_ID` source column is used for sector comparisons. Sector membership is never inferred from identifier text.
- The proportional-hazards assumption cannot be tested before fitting. Notebook 09 exports a Schoenfeld-residual time-trend screen, and notebook 10 visualizes it. A meaningful violation calls for a time interaction or a revised stratification—not merely a prettier plot.
- Both models remain observational association models. Passing these checks does not establish causal identification.
